# Day 14 — Python Concurrency for Data Science: Parallel Pipelines

**Data:** 27/10/2026 · **Fase 1 — Python Basics (Dia 14/28)**

### Contexto de aprendizado

Nos dias anteriores construí pipelines *lazy* com `itertools`/`functools` (Dia 12) e tipei esse código com `typing` (Dia 13). Hoje a pergunta é outra: **como usar mais de um núcleo (ou esperar menos por I/O) quando o pipeline cresce?**

Vou estudar quatro ferramentas da biblioteca padrão e do ecossistema científico:

| Ferramenta | Melhor para | Por quê |
| --- | --- | --- |
| `threading` / `ThreadPoolExecutor` | Tarefas **I/O-bound** (rede, disco) | Threads liberam o GIL enquanto esperam |
| `multiprocessing` / `ProcessPoolExecutor` | Tarefas **CPU-bound** em Python puro | Cada processo tem seu próprio interpretador e GIL |
| `joblib.Parallel` + `delayed` | Loops científicos, scikit-learn | API simples, backend `loky` robusto |
| `concurrent.futures` | Interface unificada | Mesma API para threads e processos |

**Datasets usados (todos públicos e reais):** seaborn (`titanic`, `penguins`, `tips`), scikit-learn (`iris`, `wine`, `breast_cancer`, `digits`, `20newsgroups`) e o NLTK Brown Corpus. Nenhum dado foi fabricado — as únicas "simulações" são *latências* artificiais com `time.sleep`, claramente identificadas.

> ⚠️ **Nota sobre benchmarks:** os tempos medidos dependem da máquina em que o notebook foi executado (número de núcleos, carga do sistema). Eles ilustram tendências, não são resultados universais.

## 0. Setup e imports

In [1]:
# Install optional dependencies if they are missing (no-op when already installed).
import importlib.util
import subprocess
import sys

for pkg in ("joblib", "nltk"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [2]:
import concurrent.futures as cf
import functools
import io
import multiprocessing as mp
import os
import platform
import re
import threading
import time
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from joblib import Parallel, delayed
from sklearn.datasets import (fetch_20newsgroups, load_breast_cancer, load_digits,
                              load_iris, load_wine)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Use "fork" explicitly on Linux so functions defined in this notebook can be sent
# to worker processes (Python 3.14 changes the POSIX default to "forkserver").
MP_CTX = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context()
N_WORKERS = min(4, os.cpu_count() or 1)
RANDOM_STATE = 42

print(f"Python      : {platform.python_version()}")
print(f"CPU cores   : {os.cpu_count()}  -> using N_WORKERS={N_WORKERS}")
print(f"Start method: {MP_CTX.get_start_method()}")
print(f"numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__} | joblib {joblib.__version__}")

Python      : 3.13.17
CPU cores   : 32  -> using N_WORKERS=4
Start method: fork
numpy 2.5.4 | pandas 3.1.0 | sklearn 1.9.1 | joblib 1.6.0


In [3]:
def split_frame(df: pd.DataFrame, n_parts: int) -> list[pd.DataFrame]:
    """Split a DataFrame into n_parts contiguous row chunks (iloc-based, numpy-version safe)."""
    bounds = np.linspace(0, len(df), n_parts + 1).astype(int)
    return [df.iloc[a:b] for a, b in zip(bounds[:-1], bounds[1:]) if b > a]


def timed(func):
    """Decorator that returns (result, elapsed_seconds) using time.perf_counter."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        return result, time.perf_counter() - start
    return wrapper

## Seção 1 — GIL: Global Interpreter Lock

O **GIL** é um *lock* do CPython que garante que **apenas uma thread execute bytecode Python por vez** dentro de um processo. Consequências práticas:

- **CPU-bound em Python puro** (loops, contagens, parsing manual): várias threads **não** aceleram — elas se revezam no GIL. Às vezes até ficam mais lentas por causa da troca de contexto.
- **I/O-bound** (esperar rede, disco, `time.sleep`): a thread **libera o GIL** enquanto espera, então outras threads trabalham → ganho real.
- **Bibliotecas em C** (NumPy, partes do scikit-learn) frequentemente liberam o GIL internamente.
- **Processos** (`multiprocessing`) têm, cada um, seu próprio interpretador e GIL → paralelismo real em CPU, ao custo de serializar (*pickle*) dados entre processos.

> Python 3.13 introduziu um build experimental *free-threaded* (PEP 703, `python3.13t`) sem GIL. Aqui uso o build padrão, então o GIL está ativo.

In [4]:
gil_status = getattr(sys, "_is_gil_enabled", lambda: True)()
print(f"GIL enabled in this interpreter: {gil_status}")

GIL enabled in this interpreter: True


In [5]:
def cpu_bound_task(n: int) -> int:
    """Pure-Python CPU work: sum of squares with an explicit loop (holds the GIL)."""
    total = 0
    for i in range(n):
        total += i * i
    return total


def io_bound_task(delay: float) -> float:
    """Simulated I/O wait (e.g. a network call). time.sleep releases the GIL."""
    time.sleep(delay)
    return delay


CPU_N = 3_000_000
N_TASKS = 4

In [6]:
@timed
def run_sequential(func, args):
    """Run func over args one after another."""
    return [func(a) for a in args]


@timed
def run_threads(func, args):
    """Run func over args with raw threading.Thread objects."""
    results = [None] * len(args)

    def worker(idx, arg):
        results[idx] = func(arg)

    threads = [threading.Thread(target=worker, args=(i, a)) for i, a in enumerate(args)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return results


@timed
def run_processes(func, args):
    """Run func over args with a multiprocessing Pool."""
    with MP_CTX.Pool(processes=N_WORKERS) as pool:
        return pool.map(func, args)


gil_rows = []
for label, func, args in [("CPU-bound", cpu_bound_task, [CPU_N] * N_TASKS),
                          ("I/O-bound", io_bound_task, [0.5] * N_TASKS)]:
    for strategy, runner in [("sequential", run_sequential), ("threading", run_threads),
                             ("multiprocessing", run_processes)]:
        _, elapsed = runner(func, args)
        gil_rows.append({"task": label, "strategy": strategy, "seconds": round(elapsed, 3)})

gil_df = pd.DataFrame(gil_rows).pivot(index="task", columns="strategy", values="seconds")
gil_df[["sequential", "threading", "multiprocessing"]]

strategy,sequential,threading,multiprocessing
task,,,
CPU-bound,0.505,0.416,0.141
I/O-bound,2.000,0.501,0.547


In [7]:
ax = gil_df[["sequential", "threading", "multiprocessing"]].plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("seconds (lower is better)")
ax.set_title("GIL demo: threads help I/O, processes help CPU")
plt.tight_layout()
plt.show()

**Leitura do resultado:**

- No **CPU-bound**, `threading` fica próximo do sequencial (GIL serializa), enquanto `multiprocessing` divide o tempo aproximadamente pelo número de workers (menos o custo de criar processos).
- No **I/O-bound**, threads e processos reduzem o tempo de ~2 s para ~0,5 s, mas **threads são mais baratas** de criar — por isso são a escolha natural para I/O.

## Seção 2 — `threading.Thread` e `ThreadPoolExecutor` para tarefas I/O-bound

Cenário típico de Data Science: baixar vários arquivos, consultar APIs ou ler muitos CSVs. Para manter o notebook reprodutível e honesto:

1. **"Download" de dados:** carrego datasets reais do seaborn **uma vez** (com cache local) e, em seguida, simulo apenas a **latência de rede** com `time.sleep`. O conteúdo é real; só a espera é artificial.
2. **Leitura paralela de CSVs em memória:** converto os DataFrames reais para texto CSV (`io.StringIO`) e faço o *parse* em paralelo.

In [8]:
SEABORN_NAMES = ["titanic", "penguins", "tips"]
REAL_FRAMES = {name: sns.load_dataset(name) for name in SEABORN_NAMES}
for name, df in REAL_FRAMES.items():
    print(f"{name:9s} shape={df.shape}")

titanic   shape=(891, 15)
penguins  shape=(344, 7)
tips      shape=(244, 7)


In [9]:
LATENCY = {"titanic": 0.6, "penguins": 0.4, "tips": 0.5}  # simulated network latency (seconds)


def fake_download(name: str) -> pd.DataFrame:
    """Return a real seaborn dataset after a simulated network delay."""
    time.sleep(LATENCY[name])
    return REAL_FRAMES[name].copy()


# 2.1 Raw threading.Thread with a shared dict protected by a Lock
downloaded: dict[str, pd.DataFrame] = {}
lock = threading.Lock()


def thread_worker(name: str) -> None:
    """Download one dataset and store it in the shared dict under a lock."""
    df = fake_download(name)
    with lock:
        downloaded[name] = df


start = time.perf_counter()
threads = [threading.Thread(target=thread_worker, args=(n,), name=f"dl-{n}") for n in SEABORN_NAMES]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"threading.Thread: {time.perf_counter() - start:.2f}s "
      f"(sequential would be ~{sum(LATENCY.values()):.1f}s)")
print({k: v.shape for k, v in downloaded.items()})

threading.Thread: 0.60s (sequential would be ~1.5s)
{'penguins': (344, 7), 'tips': (244, 7), 'titanic': (891, 15)}


O `threading.Lock` garante que duas threads não escrevam no dicionário ao mesmo tempo. Em CPython, operações simples em `dict` já são atômicas, mas o *lock* deixa a intenção explícita e protege lógicas compostas (ler → modificar → escrever).

### `ThreadPoolExecutor`: a forma moderna

Em vez de gerenciar threads manualmente, o `ThreadPoolExecutor` mantém um *pool* reutilizável e devolve objetos `Future`. Duas formas de uso:

- `executor.map(func, iterable)` → resultados **na ordem de entrada**;
- `executor.submit(...)` + `as_completed(...)` → resultados **na ordem de conclusão**.

In [10]:
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=N_WORKERS, thread_name_prefix="dl") as ex:
    frames = dict(zip(SEABORN_NAMES, ex.map(fake_download, SEABORN_NAMES)))
print(f"executor.map    : {time.perf_counter() - start:.2f}s -> order kept: {list(frames)}")

start = time.perf_counter()
completion_order = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(fake_download, n): n for n in SEABORN_NAMES}
    for fut in as_completed(futures):
        completion_order.append(futures[fut])
print(f"as_completed    : {time.perf_counter() - start:.2f}s -> completion order: {completion_order}")

executor.map    : 0.60s -> order kept: ['titanic', 'penguins', 'tips']


as_completed    : 0.60s -> completion order: ['penguins', 'tips', 'titanic']


In [11]:
def flaky_download(name: str) -> pd.DataFrame:
    """Raise for an unknown dataset name to show how Future propagates exceptions."""
    if name not in REAL_FRAMES:
        raise KeyError(f"dataset '{name}' not available")
    return fake_download(name)


with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(flaky_download, n): n for n in ["tips", "does_not_exist", "penguins"]}
    for fut in as_completed(futures):
        name = futures[fut]
        try:
            print(f"OK   {name:15s} rows={len(fut.result())}")
        except KeyError as err:
            print(f"FAIL {name:15s} -> {err}")

FAIL does_not_exist  -> "dataset 'does_not_exist' not available"


OK   penguins        rows=344
OK   tips            rows=244


Exceções levantadas dentro de uma thread ficam guardadas no `Future` e só reaparecem quando chamamos `.result()`. Isso permite que **uma falha não derrube o lote inteiro** — essencial em pipelines de ingestão.

### Leitura paralela de "arquivos" CSV em memória

Divido cada dataset real em partes (como se fossem vários arquivos CSV particionados) e faço o *parse* em paralelo.

In [12]:
def split_to_csv_buffers(df: pd.DataFrame, n_parts: int) -> list[str]:
    """Split a DataFrame into n_parts CSV strings (simulating partitioned files)."""
    return [part.to_csv(index=False) for part in split_frame(df, n_parts)]


csv_files = {f"{name}_part{i}.csv": text
             for name, df in REAL_FRAMES.items()
             for i, text in enumerate(split_to_csv_buffers(df, 4))}
print(f"{len(csv_files)} in-memory CSV 'files'; e.g. {list(csv_files)[:3]}")


def read_csv_text(item: tuple[str, str]) -> tuple[str, pd.DataFrame]:
    """Parse one in-memory CSV file, adding a small simulated disk latency."""
    fname, text = item
    time.sleep(0.05)  # simulated disk / object-storage latency
    return fname, pd.read_csv(io.StringIO(text))


_, t_seq = timed(lambda: [read_csv_text(it) for it in csv_files.items()])()
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    parsed = dict(ex.map(read_csv_text, csv_files.items()))
t_thr = time.perf_counter() - start
print(f"sequential: {t_seq:.2f}s | ThreadPoolExecutor: {t_thr:.2f}s")

12 in-memory CSV 'files'; e.g. ['titanic_part0.csv', 'titanic_part1.csv', 'titanic_part2.csv']


sequential: 0.62s | ThreadPoolExecutor: 0.16s


In [13]:
reassembled = {name: pd.concat([parsed[f] for f in sorted(parsed) if f.startswith(name + "_")],
                               ignore_index=True)
               for name in SEABORN_NAMES}
for name in SEABORN_NAMES:
    print(f"{name:9s} original={REAL_FRAMES[name].shape} reassembled={reassembled[name].shape}")

titanic   original=(891, 15) reassembled=(891, 15)
penguins  original=(344, 7) reassembled=(344, 7)
tips      original=(244, 7) reassembled=(244, 7)


## Seção 3 — `multiprocessing.Pool` e `ProcessPoolExecutor` para tarefas CPU-bound

Agora o gargalo é **CPU em Python puro**. Exemplo: feature engineering linha a linha no `titanic` e no `penguins` — o tipo de lógica com `if/else` e manipulação de strings que não vetoriza trivialmente.

Estratégia **split → apply → combine**:
1. dividir o DataFrame em *chunks*;
2. enviar cada chunk a um processo;
3. concatenar os resultados.

> Cada chunk é serializado (*pickle*) para o processo filho e de volta. Se o trabalho por chunk for pequeno, esse custo domina e o paralelismo **perde** para o sequencial.

In [14]:
titanic = REAL_FRAMES["titanic"]
# Replicate rows (not fabricate values) so each chunk has enough CPU work to measure.
titanic_big = pd.concat([titanic] * 40, ignore_index=True)
print(f"titanic: {titanic.shape} -> replicated for benchmarking: {titanic_big.shape}")


def engineer_titanic_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Row-wise, pure-Python feature engineering on a titanic chunk (CPU-bound)."""
    out = chunk[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()
    family, fare_band, age_group, score = [], [], [], []
    for row in chunk.itertuples(index=False):
        fam = row.sibsp + row.parch + 1
        family.append(fam)
        fare = row.fare if row.fare == row.fare else 0.0
        fare_band.append("low" if fare < 10 else "mid" if fare < 50 else "high")
        age = row.age
        age_group.append("unknown" if age != age else "child" if age < 13
                         else "teen" if age < 20 else "adult" if age < 60 else "senior")
        # deliberately heavier arithmetic to make the task CPU-bound
        s = 0.0
        for k in range(1, 200):
            s += (fare % k + fam) / k
        score.append(s)
    out["family_size"] = family
    out["fare_band"] = fare_band
    out["age_group"] = age_group
    out["synthetic_score"] = score  # deterministic transformation of real columns
    return out

titanic: (891, 15) -> replicated for benchmarking: (35640, 15)


In [15]:
chunks = split_frame(titanic_big, N_WORKERS * 2)

start = time.perf_counter()
seq_result = pd.concat([engineer_titanic_chunk(c) for c in chunks], ignore_index=True)
t_seq = time.perf_counter() - start

start = time.perf_counter()
with MP_CTX.Pool(processes=N_WORKERS) as pool:
    pool_result = pd.concat(pool.map(engineer_titanic_chunk, chunks), ignore_index=True)
t_pool = time.perf_counter() - start

print(f"sequential          : {t_seq:.2f}s")
print(f"multiprocessing.Pool: {t_pool:.2f}s  (speed-up x{t_seq / t_pool:.2f})")
pd.testing.assert_frame_equal(seq_result, pool_result)
print("Results identical ✔")
pool_result.head()

sequential          : 0.56s
multiprocessing.Pool: 0.20s  (speed-up x2.78)
Results identical ✔


,survived,pclass,sex,age,fare,sibsp,parch,family_size,fare_band,age_group,synthetic_score
0,0,3,male,22.0,7.2500,1,0,2,low,adult,38.325536
1,1,1,female,38.0,71.2833,1,0,2,high,adult,116.395089
2,1,3,female,26.0,7.9250,0,0,1,low,adult,36.416801
3,1,1,female,35.0,53.1000,1,0,2,high,adult,104.604005
4,0,3,male,35.0,8.0500,0,0,1,low,adult,33.150930


In [16]:
(pool_result.drop_duplicates()
 .groupby(["age_group", "fare_band"], observed=True)["survived"]
 .agg(["mean", "count"]).round(3).head(12))

mean  count
age_group fare_band              
adult     high       0.709    110
          low        0.176    170
          mid        0.422    211
child     high       0.750      4
          low        1.000      1
          mid        0.556     63
senior    high       0.571      7
          low        0.200      5
          mid        0.154     13
teen      high       0.733     15
          low        0.359     39
          mid        0.389     36

O `drop_duplicates()` acima remove as réplicas usadas apenas para o benchmark, de modo que as estatísticas de sobrevivência reflitam o dataset original de 891 passageiros (linhas idênticas no original também são colapsadas — limitação aceitável para uma ilustração).

### `Pool.starmap` e `imap_unordered`

- `starmap` passa **múltiplos argumentos** por tarefa;
- `imap_unordered` devolve resultados assim que ficam prontos (bom para barras de progresso e economia de memória).

In [17]:
penguins = REAL_FRAMES["penguins"].dropna().reset_index(drop=True)
NUMERIC = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]


def bootstrap_mean(column: str, seed: int, n_boot: int = 2000) -> tuple[str, float, float, float]:
    """Bootstrap 95% CI of the mean for one penguins column (pure-Python resampling loop)."""
    rng = np.random.default_rng(seed)
    values = penguins[column].to_numpy()
    means = [values[rng.integers(0, len(values), len(values))].mean() for _ in range(n_boot)]
    lo, hi = np.percentile(means, [2.5, 97.5])
    return column, values.mean(), lo, hi


with MP_CTX.Pool(processes=N_WORKERS) as pool:
    ci_rows = pool.starmap(bootstrap_mean, [(c, RANDOM_STATE + i) for i, c in enumerate(NUMERIC)])
pd.DataFrame(ci_rows, columns=["feature", "mean", "ci_low", "ci_high"]).round(2)

,feature,mean,ci_low,ci_high
0,bill_length_mm,43.99,43.39,44.56
1,bill_depth_mm,17.16,16.97,17.38
2,flipper_length_mm,200.97,199.43,202.51
3,body_mass_g,4207.06,4124.07,4292.12


In [18]:
def species_profile(species: str) -> dict:
    """Compute per-species descriptive statistics for the penguins dataset."""
    sub = penguins[penguins["species"] == species]
    return {"species": species, "n": len(sub),
            **{f"{c}_median": sub[c].median() for c in NUMERIC}}


with MP_CTX.Pool(processes=N_WORKERS) as pool:
    profiles = list(pool.imap_unordered(species_profile, penguins["species"].unique()))
pd.DataFrame(profiles).sort_values("species").reset_index(drop=True)

,species,n,bill_length_mm_median,bill_depth_mm_median,flipper_length_mm_median,body_mass_g_median
0,Adelie,146,38.85,18.40,190.0,3700.0
1,Chinstrap,68,49.55,18.45,196.0,3700.0
2,Gentoo,119,47.40,15.00,216.0,5050.0


### `ProcessPoolExecutor`: mesma API do `ThreadPoolExecutor`

A grande vantagem de `concurrent.futures` é que trocar threads por processos é **uma linha**. Aqui também passo `mp_context` explicitamente para controlar o método de início.

In [19]:
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as ex:
    ppe_result = pd.concat(ex.map(engineer_titanic_chunk, chunks), ignore_index=True)
t_ppe = time.perf_counter() - start
pd.testing.assert_frame_equal(seq_result, ppe_result)
print(f"ProcessPoolExecutor: {t_ppe:.2f}s (speed-up x{t_seq / t_ppe:.2f}) — results identical ✔")

ProcessPoolExecutor: 0.23s (speed-up x2.49) — results identical ✔


In [20]:
# Effect of chunk count: too few chunks -> idle workers; too many -> pickling overhead.
chunk_rows = []
for n_chunks in [1, 2, N_WORKERS, N_WORKERS * 4, N_WORKERS * 32]:
    parts = split_frame(titanic_big, n_chunks)
    start = time.perf_counter()
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as ex:
        list(ex.map(engineer_titanic_chunk, parts))
    chunk_rows.append({"n_chunks": n_chunks, "seconds": round(time.perf_counter() - start, 3)})
pd.DataFrame(chunk_rows)

,n_chunks,seconds
0,1,0.507
1,2,0.281
2,4,0.207
3,16,0.215
4,128,0.270


## Seção 4 — `joblib.Parallel` + `delayed`

`joblib` é a biblioteca de paralelismo usada internamente pelo scikit-learn (`n_jobs=-1`). A sintaxe é:

```python
Parallel(n_jobs=4)(delayed(func)(arg) for arg in iterable)
```

- `delayed(func)(arg)` **não executa** a função — cria uma tupla `(func, args, kwargs)`;
- `Parallel` distribui essas tarefas. O backend padrão `loky` usa processos robustos e `cloudpickle` (aceita funções definidas no notebook e lambdas);
- `prefer="threads"` ou `backend="threading"` troca para threads — útil quando o trabalho é em NumPy/C que libera o GIL.

In [21]:
task = delayed(cpu_bound_task)(10)
print("delayed object:", task)
print(Parallel(n_jobs=N_WORKERS)(delayed(cpu_bound_task)(n) for n in [10, 100, 1000]))

delayed object: (<function cpu_bound_task at 0x71aea3862f20>, (10,), {})


[285, 328350, 332833500]


### 4.1 Paralelizando transformações de features (por coluna)

In [22]:
iris, wine, cancer = load_iris(as_frame=True), load_wine(as_frame=True), load_breast_cancer(as_frame=True)


def column_profile(name: str, values: np.ndarray, n_boot: int = 300) -> dict:
    """Robust profile for one feature: z-score outliers, skewness and bootstrap SE of the median."""
    rng = np.random.default_rng(RANDOM_STATE)
    z = (values - values.mean()) / values.std()
    medians = [np.median(rng.choice(values, len(values))) for _ in range(n_boot)]
    skew = float(((values - values.mean()) ** 3).mean() / values.std() ** 3)
    return {"feature": name, "outliers_|z|>3": int((np.abs(z) > 3).sum()),
            "skewness": round(skew, 3), "median_se": round(float(np.std(medians)), 4)}


X_cancer = cancer.data
start = time.perf_counter()
profiles = Parallel(n_jobs=N_WORKERS)(
    delayed(column_profile)(col, X_cancer[col].to_numpy()) for col in X_cancer.columns)
print(f"{len(profiles)} breast_cancer features profiled in {time.perf_counter() - start:.2f}s")
pd.DataFrame(profiles).sort_values("skewness", ascending=False).head(8)

30 breast_cancer features profiled in 0.10s


,feature,outliers_|z|>3,skewness,median_se
13,area error,6,5.433,0.8353
16,concavity error,6,5.097,0.0012
19,fractal dimension error,10,3.914,0.0001
12,perimeter error,8,3.435,0.0624
10,radius error,7,3.080,0.0096
14,smoothness error,7,2.308,0.0001
18,symmetry error,11,2.189,0.0003
15,compactness error,12,1.897,0.0009


In [23]:
def log_transform_if_skewed(series: pd.Series, threshold: float = 1.0) -> pd.Series:
    """Apply log1p to a non-negative feature whose skewness exceeds threshold."""
    skew = series.skew()
    if skew > threshold and (series >= 0).all():
        return np.log1p(series).rename(f"log_{series.name}")
    return series


transformed = Parallel(n_jobs=N_WORKERS, prefer="threads")(
    delayed(log_transform_if_skewed)(X_cancer[c]) for c in X_cancer.columns)
X_cancer_t = pd.concat(transformed, axis=1)
n_logged = sum(c.startswith("log_") for c in X_cancer_t.columns)
print(f"{n_logged} of {X_cancer_t.shape[1]} features log-transformed")
print("skewness before (mean |skew|):", round(X_cancer.skew().abs().mean(), 3))
print("skewness after  (mean |skew|):", round(X_cancer_t.skew().abs().mean(), 3))

22 of 30 features log-transformed
skewness before (mean |skew|): 1.74
skewness after  (mean |skew|): 1.198


Usei `prefer="threads"` aqui porque `np.log1p` e `Series.skew` rodam em código C/vetorizado e o custo de copiar colunas para outros processos não compensaria.

### 4.2 Cross-validation paralela de vários modelos × datasets

Cada combinação (dataset, modelo) é uma tarefa independente — caso perfeito para paralelismo "embaraçosamente paralelo".

In [24]:
DATASETS = {"iris": (iris.data, iris.target), "wine": (wine.data, wine.target),
            "breast_cancer": (cancer.data, cancer.target)}
MODELS = {
    "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "random_forest": lambda: RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=1),
}
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def evaluate(ds_name: str, model_name: str) -> dict:
    """Run 5-fold stratified CV for one (dataset, model) pair and return summary metrics."""
    X, y = DATASETS[ds_name]
    start = time.perf_counter()
    scores = cross_val_score(MODELS[model_name](), X, y, cv=CV, scoring="accuracy", n_jobs=1)
    return {"dataset": ds_name, "model": model_name, "mean_acc": scores.mean(),
            "std_acc": scores.std(), "fit_seconds": time.perf_counter() - start}


grid = [(d, m) for d in DATASETS for m in MODELS]
_, t_cv_seq = timed(lambda: [evaluate(d, m) for d, m in grid])()
start = time.perf_counter()
cv_results = Parallel(n_jobs=N_WORKERS)(delayed(evaluate)(d, m) for d, m in grid)
t_cv_par = time.perf_counter() - start
print(f"sequential: {t_cv_seq:.2f}s | joblib (n_jobs={N_WORKERS}): {t_cv_par:.2f}s")
pd.DataFrame(cv_results).round(4)

sequential: 3.95s | joblib (n_jobs=4): 2.57s


,dataset,model,mean_acc,std_acc,fit_seconds
0,iris,logreg,0.9533,0.0452,0.0731
1,iris,random_forest,0.9600,0.0389,1.0969
2,wine,logreg,0.9833,0.0136,0.0652
3,wine,random_forest,0.9775,0.0213,1.1052
4,breast_cancer,logreg,0.9737,0.0166,0.0402
5,breast_cancer,random_forest,0.9543,0.0102,1.5531


**Cuidado com paralelismo aninhado:** dentro de cada tarefa usei `n_jobs=1` no `RandomForestClassifier` e no `cross_val_score`. Se cada tarefa externa também abrisse vários processos, teríamos *oversubscription* (mais workers do que núcleos), o que costuma deixar tudo **mais lento**.

### 4.3 `return_as="generator"` e cache com `joblib.Memory`

In [25]:
gen = Parallel(n_jobs=N_WORKERS, return_as="generator")(
    delayed(evaluate)("wine", m) for m in MODELS)
for res in gen:  # results are consumed lazily, in submission order
    print(f"{res['model']:14s} acc={res['mean_acc']:.4f}")

logreg         acc=0.9833


random_forest  acc=0.9775


In [26]:
import tempfile

cache_dir = tempfile.mkdtemp(prefix="joblib_cache_")
memory = joblib.Memory(cache_dir, verbose=0)
cached_evaluate = memory.cache(evaluate)

_, t_first = timed(cached_evaluate)("breast_cancer", "random_forest")
_, t_second = timed(cached_evaluate)("breast_cancer", "random_forest")
print(f"first call: {t_first:.3f}s | cached call: {t_second:.4f}s")
memory.clear(warn=False)

first call: 1.474s | cached call: 0.0004s


## Seção 5 — Benchmark comparativo

Comparo as quatro estratégias em **dois tipos de carga**, medindo com `time.perf_counter` (relógio monotônico de alta resolução) e repetindo cada medição 3 vezes (reporto a mediana):

1. **CPU-bound:** feature engineering linha a linha no `titanic` replicado (Seção 3);
2. **I/O-bound:** parsing dos CSVs em memória com latência simulada (Seção 2).

In [27]:
def bench(strategy: str, func, items, repeats: int = 3) -> float:
    """Median wall-clock seconds for applying func to items with a given strategy."""
    def once():
        start = time.perf_counter()
        if strategy == "sequential":
            list(map(func, items))
        elif strategy == "threading":
            with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
                list(ex.map(func, items))
        elif strategy == "multiprocessing":
            with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as ex:
                list(ex.map(func, items))
        elif strategy == "joblib":
            Parallel(n_jobs=N_WORKERS)(delayed(func)(it) for it in items)
        else:
            raise ValueError(strategy)
        return time.perf_counter() - start
    return float(np.median([once() for _ in range(repeats)]))


STRATEGIES = ["sequential", "threading", "multiprocessing", "joblib"]
WORKLOADS = {"CPU-bound (titanic FE)": (engineer_titanic_chunk, chunks),
             "I/O-bound (CSV read)": (read_csv_text, list(csv_files.items()))}

bench_rows = [{"workload": w, "strategy": s, "seconds": bench(s, f, items)}
              for w, (f, items) in WORKLOADS.items() for s in STRATEGIES]
bench_df = pd.DataFrame(bench_rows)
bench_df["speedup_vs_seq"] = bench_df.groupby("workload")["seconds"].transform(
    lambda s: s.iloc[0] / s)
bench_df.round(3)

,workload,strategy,seconds,speedup_vs_seq
0,CPU-bound (titanic FE),sequential,0.577,1.000
1,CPU-bound (titanic FE),threading,0.611,0.944
2,CPU-bound (titanic FE),multiprocessing,0.259,2.223
3,CPU-bound (titanic FE),joblib,0.187,3.086
4,I/O-bound (CSV read),sequential,0.618,1.000
5,I/O-bound (CSV read),threading,0.159,3.897
6,I/O-bound (CSV read),multiprocessing,0.186,3.319
7,I/O-bound (CSV read),joblib,0.162,3.813


In [28]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (workload, sub) in zip(axes, bench_df.groupby("workload", sort=False)):
    sns.barplot(data=sub, x="strategy", y="seconds", hue="strategy", ax=ax, legend=False,
                palette="viridis", order=STRATEGIES)
    for p, sp in zip(ax.patches, sub.set_index("strategy").loc[STRATEGIES, "speedup_vs_seq"]):
        ax.annotate(f"x{sp:.1f}", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=9)
    ax.set_title(workload)
    ax.set_xlabel("")
fig.suptitle(f"Sequential vs threading vs multiprocessing vs joblib ({N_WORKERS} workers)")
plt.tight_layout()
plt.show()

**Interpretação esperada (confira com os números acima):**

- **CPU-bound:** `multiprocessing` e `joblib` ganham; `threading` fica perto do sequencial por causa do GIL.
- **I/O-bound:** todas as opções paralelas ganham; `threading` é a mais leve. `joblib` com processos paga o custo extra de iniciar workers (o `loky` reaproveita workers entre chamadas, o que reduz esse custo nas repetições).
- O **speed-up nunca é linear**: há custo de serialização, criação de processos e partes sequenciais (Lei de Amdahl).

## Seção 6 — Aplicação NLP/Educação: processamento paralelo de corpus

Aqui junto concorrência com meu interesse em **NLP aplicado ao ensino de inglês**. Uso o **Brown Corpus** (NLTK), o primeiro corpus eletrônico de 1 milhão de palavras do inglês americano (1961), dividido em 15 gêneros — ótimo para comparar **complexidade lexical por gênero textual** (registro), um tema central da sociolinguística.

Para cada documento calculo, em paralelo com `concurrent.futures`:

- **tokenização** e frequência de palavras;
- **TTR** (*type-token ratio*) e **MATTR** (TTR com janela móvel, menos sensível ao tamanho do texto);
- **comprimento médio de palavra e de sentença**;
- **proporção de palavras longas** (≥ 7 letras) como proxy simples de dificuldade.

> Essas métricas são **heurísticas descritivas**, não níveis CEFR oficiais nem medidas validadas de dificuldade de leitura.

In [29]:
nltk.download("brown", quiet=True)
from nltk.corpus import brown

fileids = brown.fileids()
print(f"Brown Corpus: {len(fileids)} documents, {len(brown.categories())} categories")
print(brown.categories())

Brown Corpus: 500 documents, 15 categories
['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']


In [30]:
TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")


def tokenize(text: str) -> list[str]:
    """Lowercase regex tokenizer keeping internal apostrophes (e.g. don't)."""
    return TOKEN_RE.findall(text.lower())


def mattr(tokens: list[str], window: int = 100) -> float:
    """Moving-average type-token ratio (Covington & McFall, 2010)."""
    if len(tokens) < window:
        return len(set(tokens)) / max(len(tokens), 1)
    counts = Counter(tokens[:window])
    ratios = [len(counts) / window]
    for i in range(window, len(tokens)):
        counts[tokens[i]] += 1
        old = tokens[i - window]
        counts[old] -= 1
        if counts[old] == 0:
            del counts[old]
        ratios.append(len(counts) / window)
    return sum(ratios) / len(ratios)


def analyze_document(fileid: str) -> dict:
    """Tokenize one Brown document and compute lexical-complexity indicators."""
    sents = brown.sents(fileid)
    text = " ".join(" ".join(s) for s in sents)
    tokens = tokenize(text)
    freq = Counter(tokens)
    n = len(tokens)
    return {
        "fileid": fileid,
        "category": brown.categories(fileid)[0],
        "tokens": n,
        "types": len(freq),
        "ttr": len(freq) / n,
        "mattr": mattr(tokens),
        "mean_word_len": sum(map(len, tokens)) / n,
        "mean_sent_len": n / len(sents),
        "long_word_ratio": sum(len(t) >= 7 for t in tokens) / n,
        "freq": freq,
    }


print({k: (round(v, 3) if isinstance(v, float) else v)
       for k, v in analyze_document(fileids[0]).items() if k != "freq"})

{'fileid': 'ca01', 'category': 'news', 'tokens': 1967, 'types': 770, 'ttr': 0.391, 'mattr': 0.716, 'mean_word_len': 4.99, 'mean_sent_len': 20.071, 'long_word_ratio': 0.271}


In [31]:
brown.ensure_loaded()  # load the lazy corpus reader before forking workers

start = time.perf_counter()
seq_docs = [analyze_document(f) for f in fileids]
t_nlp_seq = time.perf_counter() - start

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as ex:
    par_docs = list(ex.map(analyze_document, fileids, chunksize=10))
t_nlp_proc = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    list(ex.map(analyze_document, fileids))
t_nlp_thr = time.perf_counter() - start

assert [d["types"] for d in seq_docs] == [d["types"] for d in par_docs]
print(f"500 documents | sequential {t_nlp_seq:.2f}s | threads {t_nlp_thr:.2f}s | "
      f"processes {t_nlp_proc:.2f}s (x{t_nlp_seq / t_nlp_proc:.2f})")

500 documents | sequential 2.29s | threads 11.74s | processes 0.65s (x3.55)


O argumento `chunksize=10` agrupa documentos por envio, reduzindo o número de viagens processo-pai ↔ filho. Tokenização com regex e contagem com `Counter` são majoritariamente Python puro → **processos** vencem threads. Nesta execução as threads ficaram até **mais lentas que o sequencial**: além do GIL, há disputa entre threads pelo leitor de corpus do NLTK, que não foi projetado para acesso concorrente. Lição: threads em tarefas CPU-bound podem piorar o desempenho, não apenas deixar de ajudar.

### Agregando frequências: *map-reduce* com `Counter`

In [32]:
doc_df = pd.DataFrame([{k: v for k, v in d.items() if k != "freq"} for d in par_docs])

# Reduce step: merge per-document Counters (Counter supports +).
global_freq = functools.reduce(lambda a, b: a + b, (d["freq"] for d in par_docs), Counter())
print(f"Corpus tokens: {sum(global_freq.values()):,} | types: {len(global_freq):,}")
print("Top 15:", global_freq.most_common(15))

Corpus tokens: 1,013,655 | types: 43,199
Top 15: [('the', 70003), ('of', 36473), ('and', 28935), ('to', 26247), ('a', 23510), ('in', 21420), ('that', 10596), ('is', 10109), ('was', 9815), ('he', 9548), ('for', 9500), ('it', 8771), ('with', 7290), ('as', 7255), ('his', 6999)]


In [33]:
genre_profile = (doc_df.groupby("category")
                 .agg(docs=("fileid", "count"), mattr=("mattr", "mean"),
                      mean_word_len=("mean_word_len", "mean"),
                      mean_sent_len=("mean_sent_len", "mean"),
                      long_word_ratio=("long_word_ratio", "mean"))
                 .sort_values("mattr", ascending=False).round(3))
genre_profile

,docs,mattr,mean_word_len,mean_sent_len,long_word_ratio
category,,,,,
reviews,17,0.749,4.798,20.477,0.248
humor,9,0.744,4.508,18.728,0.205
adventure,29,0.735,4.285,13.138,0.157
science_fiction,6,0.733,4.505,13.097,0.193
editorial,27,0.730,4.750,18.660,0.244
mystery,24,0.727,4.257,12.686,0.155
news,44,0.727,4.780,19.306,0.246
romance,29,0.726,4.265,14.387,0.157
fiction,29,0.723,4.328,14.936,0.165


In [34]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
order = genre_profile.index
sns.boxplot(data=doc_df, y="category", x="mattr", order=order, ax=axes[0], color="#8ecae6")
axes[0].set_title("Lexical diversity (MATTR, window=100) by Brown genre")
sns.scatterplot(data=genre_profile, x="mean_sent_len", y="long_word_ratio", size="docs",
                ax=axes[1], legend=False)
for genre, row in genre_profile.iterrows():
    axes[1].annotate(genre, (row["mean_sent_len"], row["long_word_ratio"]), fontsize=8)
axes[1].set_title("Sentence length vs long-word ratio (genre means)")
plt.tight_layout()
plt.show()

**Leitura linguística (descritiva):** gêneros informativos/acadêmicos (por exemplo `learned`, `government`) apresentaram, nesta execução, as maiores proporções de palavras longas e sentenças longas, mas a **menor** diversidade lexical (MATTR) — provavelmente porque repetem terminologia técnica. Já a ficção (`romance`, `mystery`, `adventure`) mostrou sentenças mais curtas e palavras mais curtas. Ou seja, "difícil" não é uma dimensão única: comprimento e diversidade podem apontar em direções opostas — marcas de **registro** que um professor pode usar para graduar textos de leitura. Nada disso substitui uma avaliação de legibilidade validada.

### Corpus extra: 20 Newsgroups (scikit-learn) com `ThreadPoolExecutor`

O download do `fetch_20newsgroups` é I/O-bound e pode falhar sem internet; por isso trato a exceção e sigo adiante.

In [35]:
try:
    news = fetch_20newsgroups(subset="train", categories=["sci.med", "rec.sport.baseball",
                                                           "comp.graphics", "talk.politics.misc"],
                              remove=("headers", "footers", "quotes"), random_state=RANDOM_STATE)
    news_df = pd.DataFrame({"text": news.data,
                            "group": [news.target_names[t] for t in news.target]})
    news_df = news_df[news_df["text"].str.split().str.len() >= 50].reset_index(drop=True)
except Exception as err:  # offline environment
    print("20 Newsgroups unavailable:", err)
    news_df = pd.DataFrame(columns=["text", "group"])
print(news_df.shape)

(1489, 2)


In [36]:
def text_metrics(text: str) -> dict:
    """Lexical indicators for a single post (reuses tokenize and mattr)."""
    toks = tokenize(text)
    return {"tokens": len(toks), "mattr": mattr(toks, window=50),
            "long_word_ratio": sum(len(t) >= 7 for t in toks) / max(len(toks), 1)}


if len(news_df):
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as ex:
        metrics = list(ex.map(text_metrics, news_df["text"], chunksize=50))
    news_metrics = pd.concat([news_df[["group"]], pd.DataFrame(metrics)], axis=1)
    display(news_metrics.groupby("group")[["tokens", "mattr", "long_word_ratio"]].median().round(3))

,tokens,mattr,long_word_ratio
group,,,
comp.graphics,96.0,0.812,0.193
rec.sport.baseball,109.0,0.824,0.163
sci.med,127.0,0.824,0.222
talk.politics.misc,147.0,0.820,0.208


## Seção 7 — Pipeline completo: ETL paralelo

Junto tudo num pipeline **Extract → Transform → Load/Aggregate**, escolhendo a ferramenta certa para cada etapa:

| Etapa | Natureza | Ferramenta |
| --- | --- | --- |
| **Extract:** carregar 7 datasets (seaborn + sklearn) | I/O (cache local/rede) | `ThreadPoolExecutor` |
| **Transform:** perfil estatístico + bootstrap por dataset | CPU | `ProcessPoolExecutor` |
| **Model:** baseline com CV por dataset de classificação | CPU (sklearn) | `joblib.Parallel` |
| **Aggregate:** consolidar num único relatório | leve | pandas sequencial |

In [37]:
LOADERS = {
    "titanic": lambda: sns.load_dataset("titanic"),
    "penguins": lambda: sns.load_dataset("penguins"),
    "tips": lambda: sns.load_dataset("tips"),
    "iris": lambda: load_iris(as_frame=True).frame,
    "wine": lambda: load_wine(as_frame=True).frame,
    "breast_cancer": lambda: load_breast_cancer(as_frame=True).frame,
    "digits": lambda: load_digits(as_frame=True).frame,
}
TARGETS = {"titanic": "survived", "penguins": "species", "tips": "time", "iris": "target",
           "wine": "target", "breast_cancer": "target", "digits": "target"}


def extract(name: str) -> tuple[str, pd.DataFrame, float]:
    """Load one dataset and report how long it took."""
    start = time.perf_counter()
    df = LOADERS[name]()
    return name, df, time.perf_counter() - start

In [38]:
def transform(name: str, df: pd.DataFrame) -> dict:
    """Clean a dataset and compute a numeric profile with a bootstrap CI of the mean missing rate."""
    target = TARGETS[name]
    num = df.drop(columns=[target]).select_dtypes("number")
    missing_by_row = df.isna().mean(axis=1).to_numpy()
    rng = np.random.default_rng(RANDOM_STATE)
    boot = [missing_by_row[rng.integers(0, len(df), len(df))].mean() for _ in range(1000)]
    clean = df.dropna(subset=[target])
    return {
        "dataset": name, "rows": len(df), "cols": df.shape[1],
        "numeric_features": num.shape[1],
        "missing_pct": round(100 * df.isna().to_numpy().mean(), 2),
        "missing_ci95": tuple(np.round(100 * np.percentile(boot, [2.5, 97.5]), 2)),
        "n_classes": clean[target].nunique(),
        "majority_class_pct": round(100 * clean[target].value_counts(normalize=True).iloc[0], 1),
        "mean_abs_corr": round(float(num.corr().abs().where(
            ~np.eye(num.shape[1], dtype=bool)).stack().mean()), 3) if num.shape[1] > 1 else np.nan,
    }


def model_baseline(name: str, df: pd.DataFrame) -> dict:
    """5-fold CV accuracy of a scaled logistic regression on numeric features."""
    target = TARGETS[name]
    data = df.dropna(subset=[target])
    X = data.drop(columns=[target]).select_dtypes("number")
    X = X.fillna(X.median())
    y = data[target].astype(str)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    scores = cross_val_score(model, X, y, cv=CV, n_jobs=1)
    return {"dataset": name, "cv_accuracy": round(scores.mean(), 4), "cv_std": round(scores.std(), 4)}

In [39]:
def run_parallel_etl() -> tuple[pd.DataFrame, dict]:
    """Run the full Extract -> Transform -> Model -> Aggregate pipeline and time each stage."""
    timings = {}

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=len(LOADERS)) as ex:
        extracted = {name: df for name, df, _ in ex.map(extract, LOADERS)}
    timings["extract (threads)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as ex:
        futures = [ex.submit(transform, n, df) for n, df in extracted.items()]
        profiles = [f.result() for f in futures]
    timings["transform (processes)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    baselines = Parallel(n_jobs=N_WORKERS)(
        delayed(model_baseline)(n, df) for n, df in extracted.items())
    timings["model (joblib)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    report = (pd.DataFrame(profiles).merge(pd.DataFrame(baselines), on="dataset")
              .assign(lift_over_majority=lambda d: (100 * d["cv_accuracy"]
                                                    - d["majority_class_pct"]).round(1))
              .sort_values("cv_accuracy", ascending=False).reset_index(drop=True))
    timings["aggregate (pandas)"] = time.perf_counter() - t0
    return report, timings


etl_report, etl_timings = run_parallel_etl()
print({k: round(v, 3) for k, v in etl_timings.items()}, "| total:", round(sum(etl_timings.values()), 2), "s")
etl_report

{'extract (threads)': 0.025, 'transform (processes)': 0.109, 'model (joblib)': 0.152, 'aggregate (pandas)': 0.003} | total: 0.29 s


,dataset,rows,cols,numeric_features,missing_pct,missing_ci95,n_classes,majority_class_pct,mean_abs_corr,cv_accuracy,cv_std,lift_over_majority
0,wine,178,14,13,0.00,"(0.0, 0.0)",3,39.9,0.305,0.9833,0.0136,58.4
1,penguins,344,7,4,0.79,"(0.29, 1.5)",3,44.2,0.569,0.9797,0.0116,53.8
2,breast_cancer,569,31,30,0.00,"(0.0, 0.0)",2,62.7,0.395,0.9737,0.0166,34.7
3,digits,1797,65,64,0.00,"(0.0, 0.0)",10,10.2,0.124,0.9711,0.0037,86.9
4,iris,150,5,4,0.00,"(0.0, 0.0)",3,33.3,0.594,0.9533,0.0452,62.0
5,tips,244,7,3,0.00,"(0.0, 0.0)",2,72.1,0.588,0.7213,0.0088,0.0
6,titanic,891,15,5,6.50,"(6.25, 6.77)",2,61.6,0.240,0.6981,0.0147,8.2


In [40]:
def run_sequential_etl() -> float:
    """Same pipeline without any concurrency, for comparison."""
    start = time.perf_counter()
    data = {n: LOADERS[n]() for n in LOADERS}
    [transform(n, df) for n, df in data.items()]
    [model_baseline(n, df) for n, df in data.items()]
    return time.perf_counter() - start


t_etl_seq = run_sequential_etl()
t_etl_par = sum(etl_timings.values())
print(f"Sequential ETL: {t_etl_seq:.2f}s | Parallel ETL: {t_etl_par:.2f}s | speed-up x{t_etl_seq / t_etl_par:.2f}")

Sequential ETL: 0.55s | Parallel ETL: 0.29s | speed-up x1.90


In [41]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.Series(etl_timings).plot.barh(ax=axes[0], color="#219ebc")
axes[0].set_xlabel("seconds")
axes[0].set_title("Parallel ETL — time per stage")
sns.barplot(data=etl_report, y="dataset", x="lift_over_majority", ax=axes[1], color="#fb8500")
axes[1].set_xlabel("CV accuracy − majority-class baseline (p.p.)")
axes[1].set_title("Baseline model lift per dataset")
plt.tight_layout()
plt.show()

**Observações sobre o ETL:**

- A etapa *Extract* é rápida aqui porque os datasets já estão em cache local; com downloads reais (rede), o ganho das threads seria bem maior.
- O ganho total depende de qual etapa domina. Se o *Model* domina, é nele que vale investir em paralelismo (Lei de Amdahl).
- `lift_over_majority` mostra quanto o modelo supera "sempre chutar a classe mais frequente" — uma checagem de sanidade mais honesta que a acurácia isolada. No `tips`, prever `time` (Lunch/Dinner) a partir de variáveis numéricas é uma tarefa didática, não uma pergunta de negócio real.

### Guia rápido de decisão

```text
A tarefa espera I/O (rede, disco, API)?        -> ThreadPoolExecutor
A tarefa é CPU em Python puro?                 -> ProcessPoolExecutor / multiprocessing.Pool
É um loop científico / sklearn / NumPy?        -> joblib.Parallel (+ prefer="threads" se for código C)
A operação pode ser vetorizada com NumPy/pandas? -> VETORIZE PRIMEIRO, paralelize depois
```

## Resumo dos aprendizados

1. **GIL:** no CPython padrão, só uma thread executa bytecode Python por vez. Threads ajudam em **I/O**, processos em **CPU**. Verifiquei isso empiricamente com `time.perf_counter`.
2. **`threading` / `ThreadPoolExecutor`:** ideais para "downloads" e leitura de muitos arquivos. `Lock` protege estado compartilhado; `Future.result()` propaga exceções sem derrubar o lote; `map` preserva a ordem, `as_completed` entrega por conclusão.
3. **`multiprocessing.Pool` / `ProcessPoolExecutor`:** paralelismo real em CPU com o padrão *split → apply → combine*. O número e o tamanho dos chunks importam: chunks demais = custo de *pickle*; chunks de menos = workers ociosos. Fixar o `mp_context` evita surpresas entre versões do Python.
4. **`joblib`:** `Parallel` + `delayed` paraleliza transformações por coluna e cross-validation de vários modelos/datasets; `Memory` faz cache em disco; `return_as="generator"` economiza memória. Evitar **paralelismo aninhado** (`n_jobs=1` dentro das tarefas).
5. **Benchmark:** speed-up nunca é linear (Lei de Amdahl + overhead). Medir sempre, com repetições e mediana, antes de concluir.
6. **NLP/Educação:** processei os 500 documentos do Brown Corpus em paralelo (tokenização, `Counter`, TTR/MATTR, comprimento de palavra/sentença) e comparei a complexidade lexical por gênero — um elo direto entre concorrência e meu background em sociolinguística e registro.
7. **ETL paralelo:** cada etapa usa a ferramenta adequada (threads para extrair, processos para transformar, joblib para modelar, pandas para agregar).

### Próximos passos

- **Dia 15:** `asyncio` e `aiohttp` — concorrência assíncrona para coletar dados de APIs (por exemplo, dicionários e corpora on-line).
- Experimentar o build **free-threaded** do Python 3.13 (`python3.13t`) e repetir o benchmark da Seção 1.
- Explorar **Dask** e **Polars** para dataframes maiores que a memória com paralelismo automático.
- Levar o pipeline da Seção 6 para um analisador de textos de leitura para alunos de inglês, com métricas validadas na literatura.